In [ ]:
from google.colab import drive  # Mengimpor modul untuk mengakses Google Drive di Colab
import pandas as pd  # Mengimpor modul pandas untuk memanipulasi data
import os  # Untuk memeriksa ekstensi file

# Mount Google Drive agar dapat mengakses file yang ada di dalamnya
drive.mount('/content/drive')

# Import dataset sesuai daftar file paths yang berisi lokasi file di Google Drive
checkin = pd.read_csv("/content/drive/MyDrive/TUBES VDI/gowalla_checkins.csv") # Path untuk file checkins
frienship =   pd.read_csv("/content/drive/MyDrive/TUBES VDI/gowalla_friendship.csv") # Path untuk file friendship
spots1 = pd.read_csv("/content/drive/MyDrive/TUBES VDI/gowalla_spots_subset1.csv") # Path untuk file spots
userinfo = pd.read_csv("/content/drive/MyDrive/TUBES VDI/gowalla_userinfo.csv") # Path untuk file userinfo

#Khusus dataset subset spots2
try:
    spots2 = pd.read_csv('/content/drive/MyDrive/TUBES VDI/gowalla_spots_subset2.csv', encoding='utf-8')
except UnicodeDecodeError:
    try:
        spots2 = pd.read_csv('/content/drive/MyDrive/TUBES VDI/gowalla_spots_subset2.csv', encoding='latin-1')
        print("File was successfully read using 'latin-1' encoding.")
    except UnicodeDecodeError:
        try:
            spots2 = pd.read_csv('/content/drive/MyDrive/TUBES VDI/gowalla_spots_subset2.csv', encoding='cp1252')
            print("File was successfully read using 'cp1252' encoding.")
        except UnicodeDecodeError:
            print("Could not read file with common encodings. Please check the file's actual encoding.")



In [ ]:
# Fungsi untuk membersihkan dataset
def clean_dataset(df):
    # Menangani missing values
    # Mengisi missing values pada kolom numerik dengan rata-rata
    df = df.fillna(df.select_dtypes(include=['float64', 'int64']).mean())

    # Mengisi missing values pada kolom kategori dengan 'unknown'
    df = df.fillna('unknown')

    # Menghapus kolom 'id' jika ada
    if 'id' in df.columns:
        df = df.drop(columns=['id'])

    # Menghapus baris duplikat
    df = df.drop_duplicates()

    return df

# Menerapkan pembersihan untuk setiap dataset
checkin_clean = clean_dataset(checkin)
frienship_clean = clean_dataset(frienship)
spots1_clean = clean_dataset(spots1)
userinfo_clean = clean_dataset(userinfo)
spots2_clean = clean_dataset(spots2)

# Menampilkan hasil pembersihan untuk setiap dataset
print("Checkin Dataset Cleaned:")
print(checkin_clean.head(), "\n")

print("Friendship Dataset Cleaned:")
print(frienship_clean.head(), "\n")

print("Spots1 Dataset Cleaned:")
print(spots1_clean.head(), "\n")

print("Userinfo Dataset Cleaned:")
print(userinfo_clean.head(), "\n")

print("Spots2 Dataset Cleaned:")
print(spots2_clean.head(), "\n")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Menghitung kepadatan dengan Gaussian KDE
xy = np.vstack([spots1_clean['lng'], spots1_clean['lat']])
z = gaussian_kde(xy)(xy)

# Menambahkan kolom density ke DataFrame
subset2['density'] = z



In [ ]:
# Setup plot dengan Cartopy
fig, ax = plt.subplots(1, 1, figsize=(15, 10), subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_global()
ax.add_feature(cfeature.LAND, edgecolor='black', facecolor='lightgrey')
ax.add_feature(cfeature.BORDERS, linestyle=':', edgecolor='black')

# Buat heatmap (scatter dengan intensitas warna berdasarkan density)
sc = ax.scatter(
    subset2['lng'],
    subset2['lat'],
    c=subset2['density'],
    cmap='YlOrRd',
    s=100,
    alpha=0.8,
    transform=ccrs.PlateCarree()
)

# Tambahkan colorbar untuk intensitas
cbar = plt.colorbar(sc, ax=ax, orientation="vertical", fraction=0.03, pad=0.04)
cbar.set_label('Density', fontsize=12)

# Tambahkan detail pada peta
ax.set_title('Global Heatmap Visualization', fontsize=16)

plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Konversi 'datetime' menjadi datetime tanpa format khusus (pandas akan mengenali format ISO8601 secara otomatis)
data['datetime'] = pd.to_datetime(checkin_clean['datetime'])

# Tambahkan kolom 'month_year' untuk mengelompokkan data berdasarkan bulan dan tahun
data['month_year'] = data['datetime'].dt.to_period('M')

# Hitung jumlah entri berdasarkan 'month_year'
checkin_count_per_month = data.groupby('month_year').size().reset_index(name='count_per_month')

# Convert 'month_year' to strings before plotting
checkin_count_per_month['month_year_str'] = checkin_count_per_month['month_year'].astype(str)


In [ ]:
# Membuat line chart
plt.figure(figsize=(10, 6))
# Use the new 'month_year_str' column for the x-axis
plt.plot(checkin_count_per_month['month_year_str'], checkin_count_per_month['count_per_month'], marker='o', linestyle='-', color='blue')

# Menambahkan label, judul, dan grid
plt.title('Jumlah Check-In Per Bulan', fontsize=16)
plt.xlabel('Month-Year', fontsize=12)
plt.ylabel('Jumlah Check-In', fontsize=12)
plt.grid(True)

# Menambahkan nilai jumlah check-in di ujung garis
for i, value in enumerate(checkin_count_per_month['count_per_month']):
    plt.text(checkin_count_per_month['month_year_str'].iloc[i], value, str(value), ha='center', va='bottom')  # Use 'month_year_str' here as well

# Rotasi label sumbu x
plt.xticks(rotation=45)

# Menyesuaikan tata letak
plt.tight_layout()

# Menampilkan plot
plt.show()


In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Membaca data pengguna aktif (misalnya, berdasarkan jumlah check-in atau kriteria lainnya)
# Menentukan pengguna yang paling aktif berdasarkan jumlah check-in dan jumlah teman
active_users = userinfo_clean[['id', 'checkin_num', 'friends_count', 'places_num']]

# Menyaring pengguna dengan check-in lebih dari 100, teman lebih dari 50, dan tempat lebih dari 20
active_users_filtered = active_users[(active_users['checkin_num'] > 2000) &
                                     (active_users['friends_count'] > 500) &
                                     (active_users['places_num'] > 500)]

# Menampilkan hasil pengguna aktif
print(active_users_filtered)

# Mengambil daftar ID pengguna aktif
active_users = active_users_filtered['id'].tolist()

# Filter data pertemanan agar hanya mencakup hubungan antar pengguna aktif
filtered_friendship_df = frienship_clean[friendship_df['userid1'].isin(active_users) &
                                       frienship_clean['userid2'].isin(active_users)]

# Membangun graf sosial menggunakan NetworkX
G = nx.Graph()


# Menambahkan edges (hubungan pertemanan) ke dalam graf hanya untuk pengguna aktif
for _, row in filtered_friendship_df.iterrows():
    G.add_edge(row['userid1'], row['userid2'])

# Menambahkan atribut ukuran node berdasarkan tingkat keaktifan
node_sizes = {}
node_colors = []

for node in G.nodes():
    # Mengambil informasi pengguna aktif terkait node
    user_info = gowalla_userinfo[gowalla_userinfo['id'] == node]
    if not user_info.empty:
        # Menentukan ukuran node berdasarkan jumlah check-in, teman, dan tempat
        activity_score = (user_info['checkin_num'].values[0] * 0.5 +  # Bobot untuk check-in
                          user_info['friends_count'].values[0] * 0.3 +  # Bobot untuk teman
                          user_info['places_num'].values[0] * 0.2)  # Bobot untuk tempat
        node_sizes[node] = activity_score
    else:
        node_sizes[node] = 10  # Ukuran default jika data tidak ada

    # Menentukan warna berdasarkan keaktifan (warna yang lebih gelap untuk pengguna lebih aktif)
    color_value = min(1.0, activity_score / 1000)  # Menjaga nilai agar antara 0 dan 1
    node_colors.append((color_value, 0, 1 - color_value))  # Gradasi warna dari hijau ke merah

# Menormalisasi ukuran node untuk visualisasi
max_size = max(node_sizes.values())
node_sizes_normalized = [1000 * (size / max_size) for size in node_sizes.values()]  # Menyesuaikan ukuran node



In [ ]:
# Menampilkan beberapa informasi tentang graf
print(f"Jumlah node: {G.number_of_nodes()}")
print(f"Jumlah edge: {G.number_of_edges()}")

# Visualisasi jaringan sosial pengguna aktif dengan ukuran node yang bervariasi dan warna
plt.figure(figsize=(15, 15))
pos = nx.spring_layout(G, seed=42, k=0.15, iterations=20)  # Layout untuk penempatan node

# Menggambar graf dengan ukuran node dan warna berdasarkan tingkat keaktifan
nx.draw(G, pos, with_labels=True, node_size=node_sizes_normalized, node_color=node_colors, font_size=12,
        font_color='black', font_weight='bold', alpha=0.9, edge_color='darkgray', width=0.5)

# Menambahkan grid dan beberapa elemen visual lainnya
plt.title("Jaringan Sosial Pengguna Aktif Gowalla (Berukuran dan Berwarna Berdasarkan Keaktifan)", fontsize=18)
plt.axis('off')  # Hilangkan axis
plt.tight_layout()
plt.show()



In [ ]:
# Menghitung frekuensi kategori
category_counts = Counter(category_checkins)

# Mengurutkan kategori berdasarkan jumlah check-ins (terbesar ke terkecil) dan ambil 10 teratas
top_categories = category_counts.most_common(10)



In [ ]:
# Membuat bar chart
categories, counts = zip(*top_categories)

plt.figure(figsize=(10, 6))
plt.bar(categories, counts)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Categories')
plt.ylabel('Total Check-ins (Jt kali)')
plt.title('Top 10 Most Popular Categories Based on Total Check-ins')
plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Mengonversi kolom datetime menjadi format datetime
checkin_clean['datetime'] = pd.to_datetime(checkin_clean['datetime'])

# Menambahkan kolom 'day_of_week' (0 = Senin, 6 = Minggu)
checkin_clean['day_of_week'] = checkin_clean['datetime'].dt.dayofweek

# Kategorikan hari kerja (0-4) dan akhir pekan (5-6)
checkin_clean['is_weekend'] = checkin_clean['day_of_week'].apply(lambda x: 'Weekend' if x >= 5 else 'Weekday')


# Hitung jumlah check-in berdasarkan kategori
check_in_counts = checkin_clean['is_weekend'].value_counts()
check_in_counts



In [ ]:
# Visualisasi
plt.figure(figsize=(8, 6))
check_in_counts.plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Perbandingan Jumlah Check-in pada Akhir Pekan vs Hari Kerja')
plt.xlabel('Kategori Hari')
plt.ylabel('Jumlah Check-in')
plt.xticks(rotation=0)
plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Mengoptimalkan tipe data
userinfo_clean= userinfo_clean.astype({
   "id": "int32",
   "checkin_num": "int32",
   "pins_count": "int32"
})

# Menentukan pengguna paling aktif berdasarkan `checkin_num`
most_active_user = userinfo_clean[userinfo_clean["checkin_num"] == userinfo_clean["checkin_num"].max()]



In [ ]:
# Visualisasi bar chart untuk aktivitas pengguna
plt.figure(figsize=(10, 6))
sns.barplot(x="id", y="checkin_num", data=df.nlargest(10, "checkin_num"), palette="viridis")
plt.title("Top 10 Aktivitas Pengguna Berdasarkan Check-in")
plt.xlabel("ID Pengguna")
plt.ylabel("Jumlah Check-in")
plt.xticks(rotation=45)
plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Count occurrences of each placeid
place_counts = checkin_clean['placeid'].value_counts()

# Get the top 10 most frequent place IDs
top_10_places = place_counts.head(10)



In [ ]:
# Plot the bar chart
plt.figure(figsize=(10, 6))
top_10_places.plot(kind='bar', color='skyblue')
plt.title('Top 10 lokasi paling populer')
plt.xlabel('Place ID')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

